In [0]:
from pyspark.sql.functions import expr, regexp_replace, upper, col

df = (
    spark.read.format("excel")
    .option("inferSchema", "true")
    .option("sheetName", "법인 공유 data+Q-report data+CQIS")
    .option("dataAddress", "A1:D9999")
    .load("/Volumes/sandbox/z_yeswook_kim/v_yeswook_kim/jira1312.xlsx")
)

# 첫 행을 컬럼으로 승격
new_columns = df.first()
df = df.filter(df[0] != new_columns[0])

# 공백을 언더바로 변환한 컬럼명 생성
new_columns_underscore = [c.replace(" ", "_") for c in new_columns]
df = df.toDF(*new_columns_underscore)

df = df.withColumn(
    "mac_addr_origin",
    regexp_replace(
        upper(col("Mac_Address")),
        r"(.{2})(?!$)",
        r"$1:"
    )
)

df = df.withColumn(
    "mac_addr",
    expr("aic_data_ods.tlamp.hash_mac(mac_addr_origin)")
)

df.write.mode("overwrite").saveAsTable("adhoc.heds.heds_1312")

display(df)

In [0]:
%sql
select *
from aic_data_mart.master_tables.mac_user_master
where 1=1
  and mac_addr = '477542f6843234733f10603b55c0edb47c3a0905bcf7b0e385bcf7958523886a'

In [0]:
%sql
select b.*
from aic_data_ods.tlamp.normal_log_webos23 a
join adhoc.heds.heds_1312 b using (mac_addr)
where 1=1
  and a.date_ym = '2026-02'


In [0]:
%sql
select *
from aic_data_ods.tlamp.normal_log_webos23 a


In [0]:
%sql
select date_ym, mac_addr, round(max(accum_run_time))
from aic_data_ods.tlamp.normal_log_webos23
where 
  mac_addr in( '00a340e7715240217dc342e588c3eb47a8def724724323a6dbc9842b11cd7070'
   , 'bc8e15356a92941fc3df1d333b52d312bf3da549756644b400ec6da0a433a4e2'
   , '7eb4c0393419e3ec88ee46aee08b2b4130ef1d686928be457405dbcf23dc6fc2')
  and date_ym >= '2025-01'
  -- and context_name = 'tvpowerd'
group by date_ym, mac_addr
order by 2,1

In [0]:
%sql
select date_ym, mac_addr, round(max(accum_run_time))
from aic_data_ods.tlamp.normal_log_webos23
where 
  mac_addr in( '00a340e7715240217dc342e588c3eb47a8def724724323a6dbc9842b11cd7070'
   , 'bc8e15356a92941fc3df1d333b52d312bf3da549756644b400ec6da0a433a4e2'
   , '7eb4c0393419e3ec88ee46aee08b2b4130ef1d686928be457405dbcf23dc6fc2')
  and date_ym >= '2025-01'
  and context_name = 'tvpowerd'
group by date_ym, mac_addr
order by 2,1

In [0]:
%sql
WITH joined AS (
  SELECT
    b.*,
    a.accum_run_time,
    a.date_ym
    -- 필요하면 로그의 타임스탬프/이벤트시간 컬럼도 같이 선택 (예: a.event_ts)
  FROM aic_data_ods.tlamp.normal_log_webos23 a
  JOIN adhoc.heds.heds_1312 b
  USING (mac_addr)
  WHERE a.date_ym = '2026-02'
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY mac_addr
      ORDER BY accum_run_time DESC
    ) AS rn
  FROM joined
)
SELECT *
FROM ranked
WHERE rn = 1;

In [0]:
%sql
WITH joined AS (
  SELECT
    b.*,
    a.accum_run_time,
    a.date_ym
    -- 필요하면 로그의 타임스탬프/이벤트시간 컬럼도 같이 선택 (예: a.event_ts)
  FROM aic_data_ods.tlamp.normal_log_webos23 a
  JOIN adhoc.heds.heds_1312 b
  USING (mac_addr)
  WHERE a.date_ym = '2026-02'
    AND 
    a.context_name = 'tvpowerd'
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY mac_addr
      ORDER BY accum_run_time DESC
    ) AS rn
  FROM joined
)
SELECT *
FROM ranked
WHERE rn = 1;

In [0]:
%sql
  SELECT
    b.*,
    a.accum_run_time,
    a.date_ym
    -- 필요하면 로그의 타임스탬프/이벤트시간 컬럼도 같이 선택 (예: a.event_ts)
  FROM adhoc.heds.heds_1312 b
  LEFT JOIN aic_data_ods.tlamp.normal_log_webos23 a
  USING (mac_addr)
  WHERE a.date_ym = '2026-02'
    AND 
    a.context_name = 'tvpowerd'

In [0]:
%sql


In [0]:
%sql
WITH joined AS (
  SELECT
    b.*,
    a.accum_run_time,
    a.date_ym
    -- 필요하면 로그의 타임스탬프/이벤트시간 컬럼도 같이 선택 (예: a.event_ts)
  FROM adhoc.heds.heds_1312 b
  LEFT JOIN aic_data_ods.tlamp.normal_log_webos23 a
  USING (mac_addr)
  WHERE a.date_ym = '2026-02'
    AND 
    a.context_name = 'tvpowerd'
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY mac_addr
      ORDER BY accum_run_time DESC
    ) AS rn
  FROM joined
)
SELECT *
FROM ranked
WHERE rn = 1;

In [0]:
%sql
select distinct b.*
from aic_data_ods.tlamp.normal_log_webos23 a
join adhoc.heds.heds_1312 b using (mac_addr)
where 1=1
  and a.date_ym = '2026-02'


In [0]:
%sql
WITH base_log AS (

  SELECT
    mac_addr,
    accum_run_time,
    date_ym,
    context_name,
    'webos23' as table_nm
  FROM aic_data_ods.tlamp.normal_log_webos23
  WHERE date_ym = '2026-02'
    AND context_name = 'tvpowerd'

  UNION ALL

  SELECT
    mac_addr,
    accum_run_time,
    date_ym,
    context_name,
    'webos24' as table_nm
  FROM aic_data_ods.tlamp.normal_log_webos24
  WHERE date_ym = '2026-02'
    AND context_name = 'tvpowerd'
),

joined AS (
  SELECT
    b.*,
    a.accum_run_time,
    a.date_ym,
    a.table_nm
  FROM base_log a
  JOIN adhoc.heds.heds_1312 b
  USING (mac_addr)
),

ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY mac_addr
      ORDER BY accum_run_time DESC
    ) AS rn
  FROM joined
)

SELECT *
FROM ranked
WHERE rn = 1;

In [0]:
%sql
WITH base_log AS (

  SELECT
    mac_addr,
    accum_run_time,
    date_ym,
    context_name,
    'webos23' as table_nm
  FROM aic_data_ods.tlamp.normal_log_webos23
  WHERE date_ym = '2026-02'
    AND context_name = 'tvpowerd'

  UNION ALL

  SELECT
    mac_addr,
    accum_run_time,
    date_ym,
    context_name,
    'webos24' as table_nm
  FROM aic_data_ods.tlamp.normal_log_webos24
  WHERE date_ym = '2026-02'
    AND context_name = 'tvpowerd'

  UNION ALL

  SELECT
    mac_addr,
    accum_run_time,
    date_ym,
    context_name,
    'webos25' as table_nm
  FROM aic_data_ods.tlamp.normal_log_webos25
  WHERE date_ym = '2026-02'
    AND context_name = 'tvpowerd'
),

joined AS (
  SELECT
    b.*,
    a.accum_run_time,
    a.date_ym,
    a.table_nm
  FROM base_log a
  JOIN adhoc.heds.heds_1312 b
  USING (mac_addr)
),

ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY mac_addr
      ORDER BY accum_run_time DESC
    ) AS rn
  FROM joined
)

SELECT *
FROM ranked
WHERE rn = 1;

In [0]:
%sql
WITH b AS (
  SELECT /*+ BROADCAST */ mac_addr
  FROM adhoc.heds.heds_1312
),

base_log AS (
  SELECT a.mac_addr, a.accum_run_time, a.date_ym, 'webos23' AS table_nm
  FROM aic_data_ods.tlamp.normal_log_webos23 a
  JOIN b USING (mac_addr)
  WHERE a.context_name = 'tvpowerd'
    AND a.date_ym BETWEEN '2025-09' AND '2026-02'

  UNION ALL

  SELECT a.mac_addr, a.accum_run_time, a.date_ym, 'webos24' AS table_nm
  FROM aic_data_ods.tlamp.normal_log_webos24 a
  JOIN b USING (mac_addr)
  WHERE a.context_name = 'tvpowerd'
    AND a.date_ym BETWEEN '2025-09' AND '2026-02'

  UNION ALL

  SELECT a.mac_addr, a.accum_run_time, a.date_ym, 'webos25' AS table_nm
  FROM aic_data_ods.tlamp.normal_log_webos25 a
  JOIN b USING (mac_addr)
  WHERE a.context_name = 'tvpowerd'
    AND a.date_ym BETWEEN '2025-09' AND '2026-02'
),

agg AS (
  SELECT
    mac_addr,
    MAX(accum_run_time) AS max_accum_run_time
  FROM base_log
  GROUP BY mac_addr
),

picked AS (
  -- max 값에 해당하는 행(동률 가능)을 다시 붙여서 date_ym/table_nm 회수
  SELECT
    l.mac_addr,
    l.accum_run_time,
    l.date_ym,
    l.table_nm,
    ROW_NUMBER() OVER (
      PARTITION BY l.mac_addr
      ORDER BY l.date_ym DESC, l.table_nm DESC
    ) AS rn
  FROM base_log l
  JOIN agg a
    ON l.mac_addr = a.mac_addr
   AND l.accum_run_time = a.max_accum_run_time
)

SELECT
  h.*,
  p.accum_run_time AS max_accum_run_time,
  p.date_ym        AS max_accum_run_time_ym,
  p.table_nm       AS src_table_nm
FROM picked p
JOIN adhoc.heds.heds_1312 h USING (mac_addr)
WHERE p.rn = 1;

In [0]:
%sql
WITH base_log AS (

  SELECT
    mac_addr,
    accum_run_time,
    date_ym,
    context_name,
    'webos23' as table_nm
  FROM aic_data_ods.tlamp.normal_log_webos23
  WHERE date_ym between '2025-09' and '2026-02'
    AND context_name = 'tvpowerd'

  UNION ALL

  SELECT
    mac_addr,
    accum_run_time,
    date_ym,
    context_name,
    'webos24' as table_nm
  FROM aic_data_ods.tlamp.normal_log_webos24
  WHERE date_ym between '2025-09' and '2026-02'
    AND context_name = 'tvpowerd'

  UNION ALL

  SELECT
    mac_addr,
    accum_run_time,
    date_ym,
    context_name,
    'webos25' as table_nm
  FROM aic_data_ods.tlamp.normal_log_webos25
  WHERE date_ym between '2025-09' and '2026-02'
    AND context_name = 'tvpowerd'
),

joined AS (
  SELECT
    b.*,
    a.accum_run_time,
    a.date_ym,
    a.table_nm
  FROM base_log a
  JOIN adhoc.heds.heds_1312 b
  USING (mac_addr)
),

ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY mac_addr
      ORDER BY accum_run_time DESC
    ) AS rn
  FROM joined
)

SELECT *
FROM ranked
WHERE rn = 1;

In [0]:
%sql
WITH b AS (
  SELECT DISTINCT mac_addr
  FROM adhoc.heds.heds_1312
),

w23 AS (
  SELECT
    a.mac_addr,
    MAX(a.accum_run_time) AS max_accum_run_time
  FROM aic_data_ods.tlamp.normal_log_webos23 a
  LEFT SEMI JOIN b
    ON a.mac_addr = b.mac_addr
  WHERE a.context_name = 'tvpowerd'
    AND a.date_ym BETWEEN '2025-09' AND '2026-02'
  GROUP BY a.mac_addr
),

w24 AS (
  SELECT
    a.mac_addr,
    MAX(a.accum_run_time) AS max_accum_run_time
  FROM aic_data_ods.tlamp.normal_log_webos24 a
  LEFT SEMI JOIN b
    ON a.mac_addr = b.mac_addr
  WHERE a.context_name = 'tvpowerd'
    AND a.date_ym BETWEEN '2025-09' AND '2026-02'
  GROUP BY a.mac_addr
),

unioned AS (
  SELECT * FROM w23
  UNION ALL
  SELECT * FROM w24
),

final_max AS (
  SELECT
    mac_addr,
    MAX(max_accum_run_time) AS max_accum_run_time
  FROM unioned
  GROUP BY mac_addr
)

SELECT
  b2.*,
  f.max_accum_run_time
FROM adhoc.heds.heds_1312 b2
JOIN final_max f
  ON b2.mac_addr = f.mac_addr;

In [0]:
%sql
create table if not exists adhoc.heds.heds_1312_cl1 as 
WITH b AS (
  SELECT DISTINCT mac_addr
  FROM adhoc.heds.heds_1312
),

w23 AS (
  SELECT
    a.mac_addr,
    MAX(a.accum_run_time) AS max_accum_run_time
  FROM aic_data_ods.tlamp.normal_log_webos23 a
  LEFT SEMI JOIN b
    ON a.mac_addr = b.mac_addr
  WHERE a.context_name = 'tvpowerd'
    AND a.date_ym BETWEEN '2024-09' AND '2026-02'
  GROUP BY a.mac_addr
),

w24 AS (
  SELECT
    a.mac_addr,
    MAX(a.accum_run_time) AS max_accum_run_time
  FROM aic_data_ods.tlamp.normal_log_webos24 a
  LEFT SEMI JOIN b
    ON a.mac_addr = b.mac_addr
  WHERE a.context_name = 'tvpowerd'
    AND a.date_ym BETWEEN '2024-09' AND '2026-02'
  GROUP BY a.mac_addr
),

unioned AS (
  SELECT * FROM w23
  UNION ALL
  SELECT * FROM w24
),

final_max AS (
  SELECT
    mac_addr,
    MAX(max_accum_run_time) AS max_accum_run_time
  FROM unioned
  GROUP BY mac_addr
)

SELECT
  b2.*,
  f.max_accum_run_time
FROM adhoc.heds.heds_1312 b2
LEFT JOIN final_max f
  ON b2.mac_addr = f.mac_addr;

In [0]:
%sql
  SELECT
    *
  FROM aic_data_ods.tlamp.normal_log_webos23 a
  LEFT JOIN adhoc.heds.heds_1312 b
    ON a.mac_addr = b.mac_addr
  WHERE a.context_name = ('com.webos.service.micomservice')
    AND a.date_ym BETWEEN '2026-01' AND '2026-02'

In [0]:
%sql
create or replace table  adhoc.heds.heds_1312_cl1_v2 as 
WITH b AS (
  SELECT DISTINCT mac_addr
  FROM adhoc.heds.heds_1312
),

w23 AS (
  SELECT
    a.mac_addr,
    MAX(a.accum_run_time) AS max_accum_run_time
  FROM aic_data_ods.tlamp.normal_log_webos23 a
  LEFT SEMI JOIN b
    ON a.mac_addr = b.mac_addr
  WHERE 1=1
    and a.context_name in ('com.webos.service.micomservice', 'com.webos.service.panelcontrolle')
    AND a.date_ym BETWEEN '2024-09' AND '2026-02'
  GROUP BY a.mac_addr
),

w24 AS (
  SELECT
    a.mac_addr,
    MAX(a.accum_run_time) AS max_accum_run_time
  FROM aic_data_ods.tlamp.normal_log_webos24 a
  LEFT SEMI JOIN b
    ON a.mac_addr = b.mac_addr
  WHERE 1=1
    and a.context_name in ('com.webos.service.micomservice', 'com.webos.service.panelcontrolle')
    AND a.date_ym BETWEEN '2024-09' AND '2026-02'
  GROUP BY a.mac_addr
),

unioned AS (
  SELECT * FROM w23
  UNION ALL
  SELECT * FROM w24
),

final_max AS (
  SELECT
    mac_addr,
    MAX(max_accum_run_time) AS max_accum_run_time
  FROM unioned
  GROUP BY mac_addr
)

SELECT
  b2.*,
  f.max_accum_run_time
FROM adhoc.heds.heds_1312 b2
LEFT JOIN final_max f
  ON b2.mac_addr = f.mac_addr;

In [0]:
%sql
select *
from adhoc.heds.heds_1312_cl1_v2

In [0]:
%sql
  SELECT
    s3_import_time, he_etl_dt, log_create_time, accum_run_time, context_name, message_id
  FROM aic_data_ods.tlamp.normal_log_webos23 a
  inner join adhoc.heds.heds_1312 using (mac_addr) 
  WHERE 1=1
    -- and a.context_name in ('com.webos.service.micomservice', 'com.webos.service.panelcontrolle')
    and a.mac_addr = 'ac46241f04cdc25c6c2a165236e03769cdc8c26317f0306970bf9e0a254a1225'
    AND a.date_ym BETWEEN '2026-02' AND '2026-02'
  ORDER BY 
    1,2,3,4

In [0]:
%sql
  SELECT
    *
  FROM aic_data_ods.tlamp.normal_log_webos23 a
  inner join adhoc.heds.heds_1312 using (mac_addr) 
  WHERE a.context_name in ('com.webos.service.micomservice', 'com.webos.service.panelcontrolle')
    AND a.date_ym BETWEEN '2026-02' AND '2026-02'

In [0]:
%sql
  SELECT
    distinct a.mac_addr
  FROM aic_data_ods.tlamp.normal_log_webos23 a
  inner join adhoc.heds.heds_1312 using (mac_addr) 
  WHERE a.context_name in ('com.webos.service.micomservice', 'com.webos.service.panelcontrolle')
    AND a.date_ym BETWEEN '2026-02' AND '2026-02'

In [0]:
%sql
  SELECT
    *
  FROM aic_data_ods.tlamp.normal_log_webos23 a
  inner join adhoc.heds.heds_1312 using (mac_addr) 
  WHERE a.context_name in ('com.webos.service.micomservice', 'com.webos.service.panelcontrolle')
    AND a.date_ym BETWEEN '2026-02' AND '2026-02'

In [0]:
%sql
select *
from adhoc.heds.heds_1312_cl1
where 1=1
    and max_accum_run_time is not null 

In [0]:
%sql
select *
from adhoc.heds.heds_1312_cl1_v2
where 1=1
  and max_accum_run_time is not null 

In [0]:
조회조건 변경
- 플랫폼 webos23,24
- 기간 : 24년 9월 이후
- context_name = tv-on 기준 

In [0]:
%sql


In [0]:
%sql


In [0]:
%sql


In [0]:
%sql
select 
  date_ym, log_create_time, he_etl_dt, context_name, message_id,accum_run_time
from aic_data_ods.tlamp.normal_log_webos23
where 
  mac_addr in( '00a340e7715240217dc342e588c3eb47a8def724724323a6dbc9842b11cd7070')
  and date_ym >= '2026-01'
order by
  context_name, message_id, log_create_time, he_etl_dt, date_ym

In [0]:
%sql
select 
  date_ym, log_create_time, he_etl_dt, context_name, message_id,accum_run_time
from aic_data_ods.tlamp.normal_log_webos23
where 
  mac_addr in( '00a340e7715240217dc342e588c3eb47a8def724724323a6dbc9842b11cd7070')
  and date_ym >= '2026-01'
order by
  accum_run_time, context_name, message_id, log_create_time, he_etl_dt, date_ym